# FacadeDriver: Multi-LLM Orchestration

Companion notebook for Chapter 3 of *Eval Infrastructure for Agent Systems*.

This notebook demonstrates the FacadeDriver pattern for orchestrating multiple LLM
providers behind a single unified interface. Based on production work at Airbnb
BPI Virtual Analyst (30+ model integrations).

In [ ]:
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Optional
import time
import json

@dataclass
class LLMResponse:
    content: str
    model: str
    provider: str
    input_tokens: int
    output_tokens: int
    cost_usd: float
    latency_ms: int
    raw_metadata: dict = field(default_factory=dict)

class LLMProvider(ABC):
    @abstractmethod
    def generate(self, prompt: str, model: str, **kwargs) -> LLMResponse:
        pass

    @abstractmethod
    def list_models(self) -> list[str]:
        pass

class FacadeDriver:
    def __init__(self):
        self._providers: dict[str, LLMProvider] = {}
        self._model_to_provider: dict[str, str] = {}
        self._cost_tracker = CostTracker()

    def register_provider(self, name: str, provider: LLMProvider):
        self._providers[name] = provider
        for model in provider.list_models():
            self._model_to_provider[model] = name

    def generate(self, prompt: str, model: str, **kwargs) -> LLMResponse:
        provider_name = self._model_to_provider.get(model)
        if not provider_name:
            raise ValueError(f'Unknown model: {model}')
        provider = self._providers[provider_name]
        response = provider.generate(prompt, model, **kwargs)
        self._cost_tracker.record(response)
        return response

    def generate_with_fallback(self, prompt: str, models: list[str], **kwargs) -> LLMResponse:
        for model in models:
            try:
                return self.generate(prompt, model, **kwargs)
            except Exception as e:
                print(f'Model {model} failed: {e}')
                continue
        raise RuntimeError(f'All {len(models)} models failed')

    def list_all_models(self) -> list[str]:
        return list(self._model_to_provider.keys())

    def cost_report(self) -> dict:
        return self._cost_tracker.report()

class CostTracker:
    def __init__(self):
        self._costs: dict[str, float] = {}
        self._token_counts: dict[str, dict] = {}

    def record(self, response: LLMResponse):
        key = f'{response.provider}/{response.model}'
        self._costs[key] = self._costs.get(key, 0) + response.cost_usd
        self._token_counts.setdefault(key, {'input': 0, 'output': 0})
        self._token_counts[key]['input'] += response.input_tokens
        self._token_counts[key]['output'] += response.output_tokens

    def report(self) -> dict:
        return {
            'total_cost': sum(self._costs.values()),
            'by_provider_model': {
                k: {'cost': v, 'tokens': self._token_counts[k]}
                for k, v in self._costs.items()
            }
        }

print('FacadeDriver classes defined successfully.')

## Mock Provider for Testing

In production, you would use real provider SDKs (Azure OpenAI, AWS Bedrock, etc.).
For this notebook, we use a mock provider to demonstrate the pattern without requiring
API keys.

In [ ]:
class MockProvider(LLMProvider):
    def __init__(self, name: str, models: dict, pricing: dict):
        self.name = name
        self.models = models  # {model_name: response_text}
        self.pricing = pricing  # {model_name: {input: x, output: y}}

    def generate(self, prompt: str, model: str, **kwargs) -> LLMResponse:
        start = time.time()
        content = self.models.get(model, f'Mock response for {model}')
        input_tokens = len(prompt.split())
        output_tokens = len(content.split())
        latency = int((time.time() - start) * 1000)
        rates = self.pricing.get(model, {'input': 0, 'output': 0})
        cost = (input_tokens * rates['input'] + output_tokens * rates['output']) / 1_000_000
        return LLMResponse(
            content=content,
            model=model,
            provider=self.name,
            input_tokens=input_tokens,
            output_tokens=output_tokens,
            cost_usd=cost,
            latency_ms=latency
        )

    def list_models(self) -> list[str]:
        return list(self.models.keys())

# Register mock providers
driver = FacadeDriver()

driver.register_provider('azure_openai', MockProvider(
    name='azure_openai',
    models={'gpt-4o': 'GPT-4o response', 'gpt-4.1': 'GPT-4.1 response', 'o3': 'o3 reasoning response'},
    pricing={'gpt-4o': {'input': 2.50, 'output': 10.00}, 'gpt-4.1': {'input': 2.00, 'output': 8.00}, 'o3': {'input': 10.00, 'output': 40.00}}
))

driver.register_provider('aws_bedrock', MockProvider(
    name='aws_bedrock',
    models={'claude-4.5-sonnet': 'Claude response', 'claude-haiku': 'Haiku response'},
    pricing={'claude-4.5-sonnet': {'input': 3.00, 'output': 15.00}, 'claude-haiku': {'input': 0.80, 'output': 4.00}}
))

driver.register_provider('vllm', MockProvider(
    name='vllm',
    models={'qwen-2.5-72b': 'Qwen response', 'llama-3.3-70b': 'Llama response'},
    pricing={'qwen-2.5-72b': {'input': 0.50, 'output': 0.80}, 'llama-3.3-70b': {'input': 0.50, 'output': 0.80}}
))

print(f'Registered {len(driver.list_all_models())} models across 3 providers:')
for model in driver.list_all_models():
    print(f'  - {model}')

## Running Requests Through the FacadeDriver

In [ ]:
# Generate with a specific model
response = driver.generate('What is the capital of France?', 'gpt-4o')
print(f'Model: {response.model}')
print(f'Provider: {response.provider}')
print(f'Content: {response.content}')
print(f'Cost: ${response.cost_usd:.6f}')
print(f'Latency: {response.latency_ms}ms')
print()

# Generate with fallback chain
response = driver.generate_with_fallback(
    'Analyze this dataset',
    ['o3', 'claude-4.5-sonnet', 'gpt-4o', 'qwen-2.5-72b']
)
print(f'Fallback result: {response.provider}/{response.model}')
print()

# Cost report
report = driver.cost_report()
print(f'Total cost: ${report["total_cost"]:.6f}')
for key, info in report['by_provider_model'].items():
    print(f'  {key}: ${info["cost"]:.6f} (in:{info["tokens"]["input"]}, out:{info["tokens"]["output"]})')

## Next Steps

In production, replace the MockProvider with real provider implementations:
- AzureOpenAIProvider (using `openai` SDK with Azure endpoint)
- BedrockProvider (using `boto3` or `anthropic` SDK)
- VertexProvider (using `google-cloud-aiplatform`)
- VLLMProvider (using `vllm` or HTTP requests to vLLM server)

See Chapter 3 of the handbook for full provider implementation examples.